# 1A pilot v4 — follow-up checks (CPU only)

> **Tóm tắt (VI):** Notebook này **không chạy mô hình**. Nó chỉ đọc các file v4 đã lưu trên Drive (`1A-pilot-v4/`) và kiểm tra những điểm bất thường trước khi viết bài. Chạy trên **CPU runtime** (`Runtime → Change runtime type → CPU`), `Run all`, cho phép Drive; mất vài phút, không tốn GPU. Kết quả ghi vào `1A-pilot-v4/followup/` và không ghi đè gì của v4.

| # | Question | Why it matters for the paper |
|---|---|---|
| F1 | Do models **collapse onto one label** when diacritics are removed? Is that why bloomz-1b7/VSFC has a normalised drop above 1? | A drop > 1 means "worse than chance"; it needs an explanation, not just a number |
| F2 | Do **Vietnamese-adapted** models really lose more, once size and task are controlled, and against a general model with the **same tokenizer**? | Potentially the most interesting finding; so far only descriptive |
| F3 | Is **teen code** harmless, or does our teen-code generator just change few words? | The claim "diacritics matter, teen code does not" is only fair at an equal dose |
| F4 | Why is the coefficient of **Δ broken syllables negative**? | A sign against intuition must be explained (collinearity?) |
| F5 | Do the RQ1 numbers change if the six excluded pairs are kept, or if only models valid on **every** task are used? | Shows the validity gate does not drive the result (and checks survivorship on ViHSD) |
| F6 | The **"p = 0"** in the v4 conclusions | A rounding bug in `key_numbers.json`: the exact p-values are read here from the tables |

## Setup and inputs

Every input is a file written by the v4 notebook. The first checks make sure they are complete and that the numbers recomputed here match v4's.

In [ ]:
import sys, os, glob, json, itertools, warnings
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/llm-research"
else:
    ROOT = os.environ.get("PILOT_ROOT", "outputs")
SMOKE = bool(os.environ.get("PILOT_SMOKE"))   # local test only
SRC = f"{ROOT}/1A-pilot-v4" + ("-smoke" if SMOKE else "")
OUT = f"{SRC}/followup"
for d in ["figures", "tables"]:
    os.makedirs(f"{OUT}/{d}", exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import spearmanr
from IPython.display import display, Markdown
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
print("reading", SRC, "→ writing", OUT)

CHECKS = []
def check(name, ok, detail=""):
    CHECKS.append({"check": name, "ok": bool(ok), "detail": str(detail)[:300]})
    print(("✅ " if ok else "❌ ") + name + (f" — {detail}" if detail else ""))
    pd.DataFrame(CHECKS).drop_duplicates("check", keep="last").to_csv(f"{OUT}/checks.csv", index=False)
    return ok

def require(name, ok, detail=""):
    if not check(name, ok, detail):
        raise RuntimeError(f"Stopped: {name}. {detail}")

# the same settings as v4
SEED, B = 42, 2000
TASKS = ["vsfc", "vsmec", "vihsd", "xnli", "vimmrc"]
CONDITIONS = ["clean", "partial25", "partial50", "partial75", "strip", "teencode", "mixed"]
NOISY = CONDITIONS[1:]
DIACRITIC = ["partial25", "partial50", "partial75", "strip"]
K = {"vsfc": 3, "vsmec": 7, "vihsd": 3, "xnli": 3, "vimmrc": 4}
CHANCE = {t: 1 / K[t] for t in TASKS}
LABELS = {"vsfc": ["negative", "neutral", "positive"],
          "vsmec": ["enjoyment", "sadness", "anger", "fear", "disgust", "surprise", "other"],
          "vihsd": ["clean", "offensive", "hate"], "xnli": ["entailment", "neutral", "contradiction"],
          "vimmrc": list("ABCD")}
TASK_NAME = {"vsfc": "VSFC (sentiment)", "vsmec": "VSMEC (emotion)", "vihsd": "ViHSD (hate speech)",
             "xnli": "XNLI-vi (inference)", "vimmrc": "ViMMRC (reading)"}
def short(m):
    return (m.split("/")[1].replace("-Instruct", "").replace("-Chat", "").replace("-chat", "").replace("-mini-instruct", "-mini")
            .replace("-v0.3", ""))

FAMILIES = ["Qwen2.5", "Qwen3", "SeaLLMs-v3", "VinaLLaMA", "bloomz", "Phi-3.5", "Mistral", "Llama-3"]
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
FAM_COLOR = dict(zip(FAMILIES, PALETTE))
mpl.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 220, "font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9,
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#8a8984", "axes.labelcolor": "#2b2a28",
    "xtick.color": "#52514e", "ytick.color": "#52514e", "axes.grid": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.6,
    "legend.frameon": False, "lines.linewidth": 2, "lines.markersize": 5, "pdf.fonttype": 42,
})
FIGS = []
def save(fig, name, caption):
    fig.savefig(f"{OUT}/figures/{name}.png", bbox_inches="tight")
    fig.savefig(f"{OUT}/figures/{name}.pdf", bbox_inches="tight")
    FIGS.append((name, caption))
    plt.show()

def save_table(df, name, caption, floatfmt="%.3f"):
    df.to_csv(f"{OUT}/tables/{name}.csv")
    try:
        df.to_latex(f"{OUT}/tables/{name}.tex", float_format=floatfmt, caption=caption, label=f"tab:{name}", na_rep="--")
    except Exception as e:
        print("latex export failed:", name, repr(e)[:100])

def ci(a):
    a = np.asarray(a, float)
    a = a[np.isfinite(a)]
    return (np.percentile(a, 2.5), np.percentile(a, 97.5)) if len(a) else (np.nan, np.nan)

def boot_p(diff):
    d = diff[np.isfinite(diff)]
    return min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean())) if len(d) else np.nan

def holm(ps):
    ps = np.asarray(ps, float)
    adj, run = np.empty_like(ps), 0
    for rank, i in enumerate(np.argsort(ps)):
        run = max(run, (len(ps) - rank) * ps[i])
        adj[i] = min(1.0, run)
    return adj

def fmt_p(p):
    return "p < 1e-10" if p < 1e-10 else f"p = {p:.2g}"

KEY, LINES = {}, []   # → followup.json and followup.md

In [ ]:
MODEL_TAB = pd.read_csv(f"{SRC}/tables/table_models.csv", index_col=0)
VALID = pd.read_csv(f"{SRC}/checks/validity.csv")
S = pd.read_csv(f"{SRC}/tables/summary_all.csv")
NOISE = pd.read_csv(f"{SRC}/checks/noise.csv")
X = pd.read_csv(f"{SRC}/tables/item_level.csv.gz")
P = pd.concat([pd.read_csv(p) for p in sorted(glob.glob(f"{SRC}/preds/*.csv"))], ignore_index=True)

HUB = {short(m): m for m in S.model.unique()}
MODELS = [HUB[s] for s in MODEL_TAB.index if s in HUB]
VI = {m: bool(MODEL_TAB.loc[short(m), "vi_adapted"]) for m in MODELS}
FAM = {m: MODEL_TAB.loc[short(m), "family"] for m in MODELS}
SIZE = {m: float(MODEL_TAB.loc[short(m), "size_B"]) for m in MODELS}
OK = set(zip(VALID[VALID.valid].model, VALID[VALID.valid].task))
P["correct"] = (P.pred == P.label).astype(int)

pairs = P.groupby(["model", "task"]).ngroups
require("a prediction file for every (model, task) in validity.csv", pairs == len(VALID), f"{pairs} files, {len(VALID)} pairs")
check("all 16 models present (Llama needs the HF_TOKEN secret in v4)", len(MODELS) == 16, f"{len(MODELS)} models")
acc = P.groupby(["model", "task", "cond"]).correct.mean()
diff = (acc - S.set_index(["model", "task", "cond"]).acc.reindex(acc.index)).abs().max()
require("accuracies recomputed from preds/ match v4's summary", diff < 1e-9, f"max difference {diff:.2g}")
print(f"{len(MODELS)} models, {pairs} (model, task) pairs, {len(OK)} valid")

# the same paired bootstrap resamples as v4 (same seed, same order)
rng = np.random.default_rng(SEED)
N_ITEMS = {t: P[P.task == t].item_id.nunique() for t in TASKS}
IDX = {t: rng.integers(0, N_ITEMS[t], (B, N_ITEMS[t])) for t in TASKS}
ACC = {(m, t): g.pivot(index="item_id", columns="cond", values="correct").sort_index()[CONDITIONS].to_numpy(float)
       for (m, t), g in P.groupby(["model", "task"])}

def boot_nd(m, t, c="strip"):
    bs = ACC[(m, t)][IDX[t]].mean(1)
    j = CONDITIONS.index(c)
    return (bs[:, 0] - bs[:, j]) / np.where(bs[:, 0] - CHANCE[t] > 0.05, bs[:, 0] - CHANCE[t], np.nan)

## F1 · Do models collapse onto one label?

For every (model, task), the share of the most frequent **predicted** label (calibrated predictions, as in the main analysis) and the normalised entropy of the predictions (1 = spread evenly over all labels, 0 = always the same label), on clean and on fully stripped text. The truth is balanced, so a good model predicts each label about 1/K of the time.

**Collapse** = under `strip` one label takes ≥ 60% of the predictions **and** at least 15 points more than on clean text.

In [ ]:
rows = []
for (m, t), g in P.groupby(["model", "task"]):
    r = {"model": m, "task": t, "family": FAM[m], "valid": (m, t) in OK}
    for c in ["clean", "strip"]:
        share = g[g.cond == c].pred.value_counts(normalize=True).reindex(range(K[t]), fill_value=0)
        nz = share[share > 0]
        r[f"top_{c}"] = LABELS[t][share.idxmax()]
        r[f"top_share_{c}"] = share.max()
        r[f"entropy_{c}"] = float(-(nz * np.log(nz)).sum() / np.log(K[t]))
        r[f"acc_{c}"] = g[g.cond == c].correct.mean()
    r["norm_drop"] = S[(S.model == m) & (S.task == t) & (S.cond == "strip")].norm_drop.iloc[0]
    rows.append(r)
COL = pd.DataFrame(rows)
COL["d_top_share"] = COL.top_share_strip - COL.top_share_clean
COL["collapse"] = (COL.top_share_strip >= 0.6) & (COL.d_top_share >= 0.15)
COL["below_chance"] = COL.acc_strip < COL.task.map(CHANCE)
save_table(COL.assign(model=COL.model.map(short)).set_index(["model", "task"]), "table_f1_collapse",
           "Most frequent predicted label and prediction entropy, clean vs strip (calibrated predictions).")

V1 = COL[COL.valid]
print(f"valid pairs that collapse under strip: {V1.collapse.sum()} of {len(V1)}")
print(f"valid pairs below chance under strip: {V1.below_chance.sum()}")
show = V1[V1.collapse | V1.below_chance | (V1.norm_drop > 1)]
display(show.assign(model=show.model.map(short))[["model", "task", "top_clean", "top_share_clean", "top_strip", "top_share_strip",
                                                   "acc_clean", "acc_strip", "norm_drop"]].round(2))
rho = spearmanr(V1.d_top_share, V1.norm_drop, nan_policy="omit")[0]
print(f"Spearman ρ between the rise in the top label's share and the normalised drop (valid pairs): {rho:.2f}")
KEY["f1"] = {"collapse": int(V1.collapse.sum()), "below_chance": int(V1.below_chance.sum()), "valid_pairs": len(V1), "rho": rho}

In [ ]:
# The pairs with a normalised drop above 1: what exactly do they predict?
over = V1[V1.norm_drop > 1].sort_values("norm_drop", ascending=False).head(8)   # the worst ones
for r in over.itertuples():
    g = P[(P.model == r.model) & (P.task == r.task)]
    cm = pd.crosstab(g[g.cond == "strip"].label.map(lambda k: LABELS[r.task][k]),
                     g[g.cond == "strip"].pred.map(lambda k: LABELS[r.task][k]), rownames=["truth"], colnames=["predicted (strip)"])
    dist = pd.DataFrame({c: g[g.cond == c].pred.value_counts(normalize=True).reindex(range(K[r.task]), fill_value=0).values
                         for c in ["clean", "strip"]}, index=LABELS[r.task])
    print(f"── {short(r.model)} / {r.task}: accuracy {r.acc_clean:.2f} → {r.acc_strip:.2f} (chance {CHANCE[r.task]:.2f}), "
          f"normalised drop {r.norm_drop:.2f}")
    display(dist.round(2).rename_axis("share of predictions"))
    display(cm)
if over.empty:
    print("no valid pair has a normalised drop above 1")

In [ ]:
fig, ax = plt.subplots(figsize=(5.6, 3.8))
for fam, g in V1.groupby("family"):
    ax.scatter(g.d_top_share, g.norm_drop, s=30, color=FAM_COLOR.get(fam, "#8a8984"), label=fam, edgecolor="white", lw=0.6)
ax.axhline(1, color="#8a8984", lw=1, ls=":")
ax.axvline(0, color="#8a8984", lw=1, ls=":")
ax.set(xlabel="rise in the share of the most frequent predicted label (strip − clean)", ylabel="normalised drop (strip)",
       title=f"Losses come with predictions piling onto one label (ρ = {rho:.2f})")
ax.legend(fontsize=7, ncol=2)
save(fig, "figF1_collapse", "Valid (model, task) pairs: rise in the share of the most frequent predicted label under full stripping "
     "against the normalised drop; dotted line = back to chance.")

## F2 · Do Vietnamese-adapted models lose more?

Three tests, from weakest to strongest control:
1. **Regression over valid pairs:** normalised drop under `strip` ~ adapted + log₂(size) + task. SEs clustered by model, plus an **exact permutation test**: the "adapted" label is moved to every possible set of models of the same size, which needs no distributional assumption with only 16 models.
2. **Matched pairs, same tokenizer:** SeaLLMs-v3 is continued pre-training of Qwen2 and uses the Qwen2 tokenizer, shared with Qwen2.5. Comparing SeaLLMs-v3 with Qwen2.5 of the same size keeps tokenization identical (checked below), so a difference comes from training, not from the tokenizer. Paired bootstrap over items, Holm-corrected.
3. **Where could it come from?** Tokenizer change (Δ tokens per syllable) and surprise (Δ NLL per syllable) under `strip`, adapted vs general.

VinaLLaMA has no general counterpart here (its base, Llama-2, is not in the set), so it enters only test 1.

In [ ]:
SR = S[(S.cond == "strip") & S.valid].dropna(subset=["norm_drop"]).copy()
SR["vi"] = SR.model.map(VI).astype(int)
SR["log_size"] = np.log2(SR["size"])
tab = SR.groupby(["task", "vi"]).norm_drop.agg(["mean", "count"]).unstack("vi")
tab.columns = [f"{a} ({'adapted' if b else 'general'})" for a, b in tab.columns]
display(tab.reindex(TASKS).round(2))

f = smf.ols("norm_drop ~ vi + log_size + C(task)", data=SR).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(SR.model)[0]})
coef = f.params["vi"]

# exact permutation test over which models are "adapted"
ms = sorted(SR.model.unique())
n_vi = sum(VI[m] for m in ms)
Xd = pd.get_dummies(SR.task, drop_first=True, dtype=float).assign(const=1.0, log_size=SR.log_size.values)
y = SR.norm_drop.to_numpy()
def vi_coef(adapted):
    Z = Xd.assign(vi=SR.model.isin(adapted).astype(float).values).to_numpy()
    return np.linalg.lstsq(Z, y, rcond=None)[0][-1]
perm = np.array([vi_coef(set(c)) for c in itertools.combinations(ms, n_vi)])
p_perm = (np.abs(perm) >= abs(coef) - 1e-12).mean()
check("permutation test reproduces the OLS coefficient", abs(vi_coef({m for m in ms if VI[m]}) - coef) < 1e-9)
F2_OLS = pd.DataFrame({"coef": f.params[["vi", "log_size"]], "lo": f.conf_int().loc[["vi", "log_size"], 0],
                       "hi": f.conf_int().loc[["vi", "log_size"], 1], "p (clustered)": f.pvalues[["vi", "log_size"]]})
F2_OLS.loc["vi", "p (exact permutation)"] = p_perm
save_table(F2_OLS, "table_f2_regression", "Normalised drop under strip ~ Vietnamese-adapted + log2(size) + task, valid pairs; "
           "SE clustered by model; exact permutation test over the adapted label.")
display(F2_OLS.round(3))
print(f"adapted models: {[short(m) for m in ms if VI[m]]}; {len(perm)} possible assignments of the label")
KEY["f2_regression"] = {"coef": coef, "lo": F2_OLS.loc["vi", "lo"], "hi": F2_OLS.loc["vi", "hi"],
                        "p_cluster": F2_OLS.loc["vi", "p (clustered)"], "p_perm": p_perm, "n_assignments": len(perm)}

In [ ]:
MATCH = [("SeaLLMs/SeaLLMs-v3-1.5B-Chat", "Qwen/Qwen2.5-1.5B-Instruct"), ("SeaLLMs/SeaLLMs-v3-7B-Chat", "Qwen/Qwen2.5-7B-Instruct")]
tokc = X.drop_duplicates("model").set_index("model").tok_class
rows = []
for a, g_ in MATCH:
    if a not in MODELS or g_ not in MODELS:
        continue
    if a in tokc and g_ in tokc:
        check(f"{short(a)} and {short(g_)} tokenize identically", tokc[a] == tokc[g_])
    for t in TASKS:
        if (a, t) not in OK or (g_, t) not in OK:
            continue
        d = boot_nd(a, t) - boot_nd(g_, t)
        lo, hi = ci(d)
        nd = S[(S.cond == "strip") & (S.task == t)].set_index("model").norm_drop
        rows.append({"adapted": short(a), "general": short(g_), "task": t, "nd_adapted": nd[a], "nd_general": nd[g_],
                     "diff (adapted − general)": nd[a] - nd[g_], "lo": lo, "hi": hi, "p": boot_p(d)})
MT = pd.DataFrame(rows, columns=["adapted", "general", "task", "nd_adapted", "nd_general", "diff (adapted − general)", "lo", "hi", "p"])
if len(MT):
    MT["p_holm"] = holm(MT.p.values)
    save_table(MT.set_index(["adapted", "task"]), "table_f2_matched",
               "SeaLLMs-v3 vs Qwen2.5 of the same size (same tokenizer): normalised drop under strip, paired bootstrap, Holm-corrected.")
    display(MT.round(3))
KEY["f2_matched"] = MT.to_dict("records")

mech = SR.groupby("vi")[["d_fertility", "d_broken", "d_nll_syl", "norm_drop"]].mean().rename(index={0: "general", 1: "adapted"})
save_table(mech, "table_f2_mechanism", "Under strip, valid pairs: change in tokens per syllable, broken syllables, NLL per syllable, and normalised drop.")
display(mech.round(3))
KEY["f2_mechanism"] = mech.to_dict("index")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), gridspec_kw={"width_ratios": [1, 1.2]})
ax = axes[0]
pm = SR.groupby("model").agg(nd=("norm_drop", "mean"), n=("task", "size")).reset_index()
for r in pm.itertuples():
    ax.scatter(SIZE[r.model], r.nd, s=40, marker="D" if VI[r.model] else "o", color=FAM_COLOR.get(FAM[r.model], "#8a8984"),
               edgecolor="#2b2a28" if VI[r.model] else "white", lw=0.8, zorder=3)
    ax.annotate(short(r.model), (SIZE[r.model], r.nd), fontsize=6, xytext=(4, 2), textcoords="offset points", color="#52514e")
ax.scatter([], [], marker="D", color="#cfcec8", edgecolor="#2b2a28", label="Vietnamese-adapted")
ax.scatter([], [], marker="o", color="#cfcec8", label="general")
ax.set(xscale="log", xticks=[1.5, 3, 7], xticklabels=["1.5", "3", "7"], xlabel="parameters (B)",
       ylabel="mean normalised drop (strip, valid tasks)", title="a. Per model")
ax.xaxis.set_minor_formatter(mpl.ticker.NullFormatter())
ax.legend(fontsize=7)
ax = axes[1]
if len(MT):
    lab = [f"{r.adapted} vs {r.general} · {r.task}" for r in MT.itertuples()]
    y_ = np.arange(len(MT))
    ax.errorbar(MT["diff (adapted − general)"], y_, xerr=[MT["diff (adapted − general)"] - MT.lo, MT.hi - MT["diff (adapted − general)"]],
                fmt="o", color=FAM_COLOR["SeaLLMs-v3"], ms=4, lw=1.2)
    ax.axvline(0, color="#8a8984", lw=1)
    ax.set(yticks=y_, yticklabels=lab, xlabel="difference in normalised drop (adapted − general), 95% CI",
           title="b. Same tokenizer: SeaLLMs-v3 vs Qwen2.5")
    ax.invert_yaxis()
plt.tight_layout()
save(fig, "figF2_adapted", "a. Mean normalised drop under strip per model (diamonds: Vietnamese-adapted); "
     "b. SeaLLMs-v3 minus Qwen2.5 of the same size, same tokenizer, paired bootstrap 95% CI.")

## F3 · Teen code at an equal dose

Our teen-code rule set has 27 common substitutions (không → ko, được → dc, …), so it changes far fewer words than stripping. The fair question is: **per word changed**, is teen code less harmful than missing diacritics? For each task, the diacritic conditions give a dose–response curve (share of words changed → mean normalised drop over valid models); the teen-code point is compared with that curve at the same share of words changed.

In [ ]:
dose = NOISE[["task", "cond", "syllables changed"]].rename(columns={"syllables changed": "changed"})
mean_nd = S[S.valid & (S.cond != "clean")].groupby(["task", "cond"]).norm_drop.mean().rename("mean_norm_drop").reset_index()
D = dose.merge(mean_nd, on=["task", "cond"])
rows = []
for t in TASKS:
    d = D[D.task == t].set_index("cond")
    if "teencode" not in d.index or not set(DIACRITIC) <= set(d.index):
        continue
    curve = d.loc[DIACRITIC].sort_values("changed")
    xs, ys = np.r_[0, curve.changed], np.r_[0, curve.mean_norm_drop]
    x_t = d.loc["teencode", "changed"]
    rows.append({"task": t, "teencode: words changed": x_t, "strip: words changed": d.loc["strip", "changed"],
                 "teencode: norm drop": d.loc["teencode", "mean_norm_drop"],
                 "diacritics at the same dose": np.interp(x_t, xs, ys),
                 "teencode per 10% words": 0.1 * d.loc["teencode", "mean_norm_drop"] / x_t if x_t > 0 else np.nan,
                 "strip per 10% words": 0.1 * d.loc["strip", "mean_norm_drop"] / d.loc["strip", "changed"]})
TEEN = pd.DataFrame(rows).set_index("task")
save_table(TEEN, "table_f3_teencode_dose", "Teen code vs diacritic removal at an equal share of words changed (mean normalised drop, valid models).")
display(TEEN.round(3))
KEY["f3"] = TEEN.to_dict("index")

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 4))
for i, t in enumerate(TASKS):
    d = D[D.task == t].set_index("cond")
    if not set(DIACRITIC) <= set(d.index):
        continue
    curve = d.loc[DIACRITIC].sort_values("changed")
    ax.plot(np.r_[0, curve.changed], np.r_[0, curve.mean_norm_drop], marker="o", color=PALETTE[i], label=TASK_NAME[t], ms=4)
    for c, mk in [("teencode", "^"), ("mixed", "s")]:
        if c in d.index:
            ax.scatter(d.loc[c, "changed"], d.loc[c, "mean_norm_drop"], marker=mk, s=55, color=PALETTE[i], edgecolor="#2b2a28", lw=0.8, zorder=3)
ax.scatter([], [], marker="^", color="#cfcec8", edgecolor="#2b2a28", label="teen code")
ax.scatter([], [], marker="s", color="#cfcec8", edgecolor="#2b2a28", label="teen code + 50% stripped")
ax.set(xlabel="share of words changed", ylabel="mean normalised drop (valid models)",
       title="Damage per word changed: diacritic removal (lines) vs teen code")
ax.legend(fontsize=7)
save(fig, "figF3_teencode_dose", "Mean normalised drop against the share of words each condition changes. Lines: 25/50/75/100% "
     "diacritic removal; triangles: teen code; squares: teen code with 50% of diacritics removed.")

## F4 · Why is the Δ broken-syllables coefficient negative?

The v4 item-level model puts Δ NLL, Δ tokens and Δ broken syllables in one regression (item × condition and model fixed effects). If Δ tokens and Δ broken syllables move together, their coefficients can split one effect into a positive and a negative part. Below: their correlation and variance inflation factors **after** the fixed effects are removed, and each predictor alone.

In [ ]:
def demean(df, cols, groups, iters=200, tol=1e-9):
    out = df[cols].astype(float).copy()
    for _ in range(iters):
        delta = 0
        for gcol in groups:
            mns = out.groupby(df[gcol].values).transform("mean")
            out -= mns
            delta = max(delta, float(np.abs(mns.to_numpy()).max()))
        if delta < tol:
            break
    return out

XS = ["d_nll", "d_tok", "d_broken"]
Xf = X.dropna(subset=["flip"] + XS).copy()
Xf = Xf[Xf.item_cond.map(Xf.item_cond.value_counts()) > 1]
DM = demean(Xf, ["flip"] + XS, ["item_cond", "model"])
Z = DM[XS] / Xf[XS].std()
print("correlation after removing the fixed effects:")
display(Z.corr().round(2))
vif = {}
for k in XS:
    others = [x for x in XS if x != k]
    r2 = sm.OLS(Z[k], Z[others]).fit().rsquared
    vif[k] = 1 / (1 - r2) if r2 < 1 else np.inf
rows = []
for name, xs in [("all three (v4)", XS), ("Δ NLL alone", ["d_nll"]), ("Δ tokens alone", ["d_tok"]),
                 ("Δ broken alone", ["d_broken"]), ("Δ tokens + Δ broken", ["d_tok", "d_broken"])]:
    r = sm.OLS(DM.flip, Z[xs]).fit(cov_type="cluster", cov_kwds={"groups": Xf.unit})
    for k in xs:
        rows.append({"model": name, "term": k, "pp_per_SD": 100 * r.params[k], "lo": 100 * r.conf_int().loc[k, 0],
                     "hi": 100 * r.conf_int().loc[k, 1], "p": r.pvalues[k], "VIF (all three)": vif[k]})
F4 = pd.DataFrame(rows)
save_table(F4.set_index(["model", "term"]), "table_f4_collinearity",
           "Item-level fixed-effects LPM: each tokenizer predictor alone and together (pp per SD); VIF after removing fixed effects.")
display(F4.round(3))
KEY["f4"] = {"corr_tok_broken": Z.d_tok.corr(Z.d_broken), "vif": vif, "rows": F4.to_dict("records")}

## F5 · Does the validity gate drive RQ1?

Mean normalised drop under `strip` per task, computed three ways: valid pairs (as in the paper), **all** pairs (the six excluded ones kept wherever the drop can be computed, i.e. clean accuracy more than 0.05 above chance), and a **balanced panel** of models that are valid on every task (no survivorship: the same models in every column). The raw drop in accuracy points needs no division by the above-chance margin, so it is shown for all pairs too.

In [ ]:
panel = [m for m in MODELS if all((m, t) in OK for t in TASKS)]
rows = []
for t in TASKS:
    st = S[(S.task == t) & (S.cond == "strip")]
    v = st[st.valid]
    rows.append({"task": t, "valid: n": len(v), "valid: mean": v.norm_drop.mean(), "valid: median": v.norm_drop.median(),
                 "all pairs: n": st.norm_drop.notna().sum(), "all pairs: mean": st.norm_drop.mean(),
                 "balanced panel: n": st.model.isin(panel).sum(), "balanced panel: mean": st[st.model.isin(panel)].norm_drop.mean(),
                 "raw drop pp, valid": 100 * v["drop"].mean(), "raw drop pp, all": 100 * st["drop"].mean()})
SENS = pd.DataFrame(rows).set_index("task")
save_table(SENS, "table_f5_sensitivity", "Normalised drop under strip: valid pairs, all pairs, and models valid on every task; raw drop in points.")
display(SENS.round(2))
print("balanced panel:", [short(m) for m in panel])
KEY["f5"] = {"panel": [short(m) for m in panel], **SENS.to_dict("index")}

## F6 · Exact p-values for the RQ3 statements

`key_numbers.json` in v4 rounded every number to 4 decimals, so a p-value of 3e-6 became 0 and the conclusions printed "p = 0". The tables keep full precision; the corrected sentences are below (the v4 notebook on GitHub is fixed for future runs).

In [ ]:
FE = pd.read_csv(f"{SRC}/tables/table_item_level_fe.csv")
rows = [{"source": f"item-level FE · {r.sample}", "term": r.term, "estimate (pp per SD)": r.pp_per_SD, "p": r.p, "report as": fmt_p(r.p)}
        for r in FE[FE["sample"].isin(["all valid models", "same tokenizer only"])].itertuples()]
if os.path.exists(f"{SRC}/tables/table_ambiguity.csv"):
    AMB = pd.read_csv(f"{SRC}/tables/table_ambiguity.csv", index_col=0)
    rows += [{"source": "ambiguity model", "term": k, "estimate (pp per SD)": r.pp_per_SD, "p": r.p, "report as": fmt_p(r.p)}
             for k, r in AMB.iterrows()]
PV = pd.DataFrame(rows)
save_table(PV.set_index(["source", "term"]), "table_f6_pvalues", "Exact p-values for the RQ3 statements.", "%.3g")
display(PV)
KEY["f6"] = PV.to_dict("records")

## Summary

Generated from the numbers above. Read it together with the tables; the interpretation notes say what each outcome means for the paper.

In [ ]:
L = ["# Pilot 1A v4 — follow-up checks (auto-generated)", ""]
f1 = KEY["f1"]
L += ["## F1 · Label collapse",
      f"- Under full stripping, {f1['collapse']} of {f1['valid_pairs']} valid (model, task) pairs collapse onto one label "
      f"(≥ 60% of predictions, +15 points vs clean); {f1['below_chance']} fall below chance.",
      f"- Across valid pairs, the rise in the top label's share correlates with the normalised drop (Spearman ρ = {f1['rho']:.2f})."]
for r in V1[V1.norm_drop > 1].sort_values("norm_drop", ascending=False).head(8).itertuples():
    L.append(f"- {short(r.model)}/{r.task} (normalised drop {r.norm_drop:.2f}): predicts '{r.top_strip}' for {r.top_share_strip:.0%} of stripped items "
             f"(clean: '{r.top_clean}' {r.top_share_clean:.0%}); accuracy {r.acc_strip:.2f} vs chance {CHANCE[r.task]:.2f}"
             + (" → a collapse, report it as such." if r.collapse else "."))
f2 = KEY["f2_regression"]
L += ["", "## F2 · Vietnamese-adapted models",
      f"- Controlling for size and task, adapted models have a normalised drop {f2['coef']:+.2f} [{f2['lo']:.2f}, {f2['hi']:.2f}] "
      f"higher than general models ({fmt_p(f2['p_cluster'])} clustered by model; exact permutation {fmt_p(f2['p_perm'])} over {f2['n_assignments']} assignments)."]
if len(MT):
    sig = MT[(MT.p_holm < 0.05)]
    L.append(f"- Same tokenizer (SeaLLMs-v3 vs Qwen2.5 of equal size): {len(sig)} of {len(MT)} task comparisons significant after Holm; "
             f"mean difference {MT['diff (adapted − general)'].mean():+.2f}, positive in {(MT['diff (adapted − general)'] > 0).sum()} of {len(MT)}.")
mech = KEY["f2_mechanism"]
if "adapted" in mech and "general" in mech:
    L.append(f"- Under strip, Δ tokens per syllable: adapted {mech['adapted']['d_fertility']:.2f} vs general {mech['general']['d_fertility']:.2f}; "
             f"Δ NLL per syllable: adapted {mech['adapted']['d_nll_syl']:.2f} vs general {mech['general']['d_nll_syl']:.2f}.")
L.append("- Interpretation: claim 'adapted models are less robust' only if the permutation test and the same-tokenizer comparison agree; "
         "otherwise report it as a descriptive pattern.")
L += ["", "## F3 · Teen code at an equal dose"]
for t, r in KEY["f3"].items():
    L.append(f"- {t}: teen code changes {r['teencode: words changed']:.0%} of words (strip {r['strip: words changed']:.0%}); "
             f"its normalised drop {r['teencode: norm drop']:.2f} vs {r['diacritics at the same dose']:.2f} for diacritic removal at the same dose.")
L.append("- Interpretation: if teen code is close to the diacritic curve, write 'teen code is as harmful per word, just rarer', not 'teen code is harmless'.")
f4 = KEY["f4"]
alone = F4[(F4.model == "Δ broken alone")].iloc[0]
joint = F4[(F4.model == "all three (v4)") & (F4.term == "d_broken")].iloc[0]
L += ["", "## F4 · Δ broken syllables",
      f"- After the fixed effects, corr(Δ tokens, Δ broken) = {f4['corr_tok_broken']:.2f}; VIF: "
      + ", ".join(f"{k} {v:.1f}" for k, v in f4["vif"].items()) + ".",
      f"- Δ broken alone: {alone.pp_per_SD:+.1f} pp per SD ({fmt_p(alone.p)}); together with the others: {joint.pp_per_SD:+.1f} pp ({fmt_p(joint.p)}).",
      ("- Interpretation: the sign changes between 'alone' and 'together' and the VIF is high: collinearity. Report the tokenizer "
       "effect as Δ tokens alone (or a combined index), not the joint coefficient."
       if np.sign(alone.pp_per_SD) != np.sign(joint.pp_per_SD) and max(f4["vif"].values()) > 5 else
       "- Interpretation: the sign is the same alone and together and the VIF is moderate, so this is not a collinearity artefact. "
       f"It is a real within-item pattern: the model whose tokenizer breaks more syllables on the same noisy item flips "
       f"{'less' if alone.pp_per_SD < 0 else 'more'}. "
       "Report it as such; the tokenizer measures do not explain the loss, Δ NLL does.")]
L += ["", "## F5 · Validity gate", f"- Balanced panel: {len(panel)} models valid on every task."]
for t, r in SENS.iterrows():
    L.append(f"- {t}: valid {r['valid: mean']:.2f} (n = {r['valid: n']:.0f}), all pairs {r['all pairs: mean']:.2f} (n = {r['all pairs: n']:.0f}), "
             f"balanced panel {r['balanced panel: mean']:.2f}; median {r['valid: median']:.2f}.")
L += ["", "## F6 · Exact p-values (replace 'p = 0')"]
L += [f"- {r['source']} · {r['term']}: {r['estimate (pp per SD)']:.1f} pp per SD, {r['report as']}" for r in KEY["f6"]]
TEXT = "\n".join(L)
open(f"{OUT}/followup.md", "w").write(TEXT)
display(Markdown(TEXT))

In [ ]:
def jsonable(o):
    if isinstance(o, dict):
        return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [jsonable(v) for v in o]
    if isinstance(o, (np.floating, float)):
        return None if not np.isfinite(o) else float(f"{float(o):.4g}")
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.bool_,)):
        return bool(o)
    return o
json.dump(jsonable(KEY), open(f"{OUT}/followup.json", "w"), ensure_ascii=False, indent=1)
pd.DataFrame(FIGS, columns=["figure", "caption"]).to_csv(f"{OUT}/figures/captions.csv", index=False)
print("checks:", sum(c["ok"] for c in CHECKS), "passed,", sum(not c["ok"] for c in CHECKS), "failed")
print("Everything is in", OUT, "— when done: Runtime → Disconnect and delete runtime.")